### Run Emotion-English-DistilRoBERTa-base on multiple text documents

In [1]:
# install the transformers library
!pip install transformers

In [2]:
# import required packages
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer

# Create class for data preparation
class SimpleDataset:
    def __init__(self, tokenized_texts):
        self.tokenized_texts = tokenized_texts

    def __len__(self):
        return len(self.tokenized_texts["input_ids"])

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.tokenized_texts.items()}

In [3]:
# load tokenizer and model, create trainer
model_name = "j-hartmann/emotion-english-distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
trainer = Trainer(model=model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### **Option 1:** Create list of texts

In [4]:
# create list of texts (can be imported from .csv, .xls etc.)
pred_texts = ['I like that', 'That is annoying', 'This is great!', 'Wouldn´t recommend it.']

### **Option 2:** Upload file to temporary Google space

In [5]:
# run cell and select file for upload
from google.colab import files
files.upload()

{}

In [7]:
# specify your filename
file_name = "/content/Twitter Emotion Classification Dataset.csv"  # note: you can right-click on your file and copy-paste the path to it here
text_column = "text"  # select the column in your csv that contains the text to be classified

# read in csv
df_pred = pd.read_csv(file_name, engine='python', on_bad_lines='skip')
pred_texts = df_pred[text_column].dropna().astype('str').tolist()

### **Option 3:** Connect to Google Drive and select file

In [ ]:
# import file stored on Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# specify your filename
file_name = "/content/YOUR_FILENAME.csv"  # note: you can right-click on your file and copy-paste the path to it here
text_column = "text"  # select the column in your csv that contains the text to be classified

# read in csv
df_pred = pd.read_csv(file_name)
pred_texts = df_pred[text_column].dropna().astype('str').tolist()

### Classify texts with model

In [8]:
# Tokenize texts and create prediction data set
tokenized_texts = tokenizer(pred_texts,truncation=True,padding=True)
pred_dataset = SimpleDataset(tokenized_texts)

In [9]:
# Run predictions
predictions = trainer.predict(pred_dataset)

In [10]:
# Transform predictions to labels
preds = predictions.predictions.argmax(-1)
labels = pd.Series(preds).map(model.config.id2label)
scores = (np.exp(predictions[0])/np.exp(predictions[0]).sum(-1,keepdims=True)).max(1)

In [11]:
# scores raw
temp = (np.exp(predictions[0])/np.exp(predictions[0]).sum(-1,keepdims=True))

In [12]:
# work in progress
# container
anger = []
disgust = []
fear = []
joy = []
neutral = []
sadness = []
surprise = []

# extract scores (as many entries as exist in pred_texts)
for i in range(len(pred_texts)):
  anger.append(temp[i][0])
  disgust.append(temp[i][1])
  fear.append(temp[i][2])
  joy.append(temp[i][3])
  neutral.append(temp[i][4])
  sadness.append(temp[i][5])
  surprise.append(temp[i][6])

In [13]:
# Create DataFrame with texts, predictions, labels, and scores
df = pd.DataFrame(list(zip(pred_texts,preds,labels,scores,  anger, disgust, fear, joy, neutral, sadness, surprise)), columns=['text','pred','label','score', 'anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise'])
df.head()

,text,pred,label,score,anger,disgust,fear,joy,neutral,sadness,surprise
0,i feel awful about it too because it s my job ...,5,sadness,0.992340,0.001399,0.001132,0.002018,0.001308,0.000596,0.992340,0.001207
1,im alone i feel awful,5,sadness,0.991029,0.000693,0.002154,0.002009,0.000879,0.002010,0.991029,0.001227
2,ive probably mentioned this before but i reall...,3,joy,0.991750,0.002016,0.000666,0.000460,0.991750,0.000906,0.002656,0.001546
3,i was feeling a little low few days back,5,sadness,0.993073,0.001643,0.000412,0.000894,0.002652,0.000612,0.993073,0.000713
4,i beleive that i am much more sensitive to oth...,3,joy,0.967344,0.020706,0.002151,0.001278,0.967344,0.001342,0.006365,0.000814


### Export results

In [14]:
# save results to csv
YOUR_FILENAME = "YOUR_FILENAME_EMOTIONS.csv"  # name your output file
df.to_csv(YOUR_FILENAME)

In [15]:
# download file
files.download(YOUR_FILENAME)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
# Define your custom label mapping
custom_label_mapping = {
    0: 'sadness',
    1: 'joy',
    2: 'love',
    3: 'anger',
    4: 'fear',
    5: 'surprise'
}

# Add original labels to the DataFrame for comparison using your custom mapping
df['original_label'] = df_pred['label'].map(custom_label_mapping).head(len(df)).tolist()
display(df.head())

,text,pred,label,score,anger,disgust,fear,joy,neutral,sadness,surprise,original_label
0,i feel awful about it too because it s my job ...,5,sadness,0.992340,0.001399,0.001132,0.002018,0.001308,0.000596,0.992340,0.001207,sadness
1,im alone i feel awful,5,sadness,0.991029,0.000693,0.002154,0.002009,0.000879,0.002010,0.991029,0.001227,sadness
2,ive probably mentioned this before but i reall...,3,joy,0.991750,0.002016,0.000666,0.000460,0.991750,0.000906,0.002656,0.001546,joy
3,i was feeling a little low few days back,5,sadness,0.993073,0.001643,0.000412,0.000894,0.002652,0.000612,0.993073,0.000713,sadness
4,i beleive that i am much more sensitive to oth...,3,joy,0.967344,0.020706,0.002151,0.001278,0.967344,0.001342,0.006365,0.000814,love


In [23]:
# Calculate accuracy
accuracy = (df['label'] == df['original_label']).mean() * 100
print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 83.85%


In [24]:
print("Model's ID to Label mapping:")
print(model.config.id2label)

Model's ID to Label mapping:
{0: 'anger', 1: 'disgust', 2: 'fear', 3: 'joy', 4: 'neutral', 5: 'sadness', 6: 'surprise'}


In [26]:
# Filter for misclassified texts
misclassified_df = df[df['label'] != df['original_label']]

# Display the first 10 misclassified examples
display(misclassified_df.head(10))

,text,pred,label,score,anger,disgust,fear,joy,neutral,sadness,surprise,original_label
4,i beleive that i am much more sensitive to oth...,3,joy,0.967344,0.020706,0.002151,0.001278,0.967344,0.001342,0.006365,0.000814,love
5,i find myself frustrated with christians becau...,0,anger,0.989957,0.989957,0.000344,0.001227,0.002308,0.000459,0.004387,0.001319,love
8,i was struggling with these awful feelings and...,5,sadness,0.520006,0.039984,0.007361,0.121700,0.306120,0.000719,0.520006,0.004109,joy
17,i feel weird knowing mine died when i wasn t a...,6,surprise,0.640611,0.003911,0.006724,0.334847,0.002536,0.004249,0.007121,0.640611,fear
19,i feel blessed everyday for our little man and...,3,joy,0.993654,0.001164,0.000360,0.000683,0.993654,0.000474,0.002271,0.001394,love
21,i stil can see kaibas face on every tv screen ...,3,joy,0.416173,0.297487,0.002469,0.014662,0.416173,0.001513,0.265708,0.001987,sadness
24,i alternate between feeling sympathetic toward...,3,joy,0.992958,0.001757,0.000836,0.000543,0.992958,0.000691,0.002501,0.000715,love
31,i pretty much waddled out of the hospital feel...,2,fear,0.621717,0.008232,0.001907,0.621717,0.003303,0.002053,0.033850,0.328938,surprise
35,i feel passionate about today because of him,3,joy,0.983760,0.009458,0.001076,0.001097,0.983760,0.000499,0.003186,0.000924,love
37,article published,4,neutral,0.651273,0.021779,0.004858,0.012384,0.145113,0.651273,0.024339,0.140253,joy
